# 02 — WAHIS Data Cleaning

## Machine Learning-Based Early Warning System for Poultry Disease Outbreaks in Nigeria

This notebook cleans and standardises the raw WAHIS quantitative surveillance dataset identified during the initial audit.

The cleaning process will:

1. Load the original raw WAHIS dataset.
2. Preserve the raw data without modification.
3. Standardise column names.
4. Convert textual unavailable markers such as `-` to missing values.
5. Convert quantitative surveillance variables to numeric data types.
6. Standardise selected categorical variables.
7. Validate year and semester fields.
8. Check for duplicate records.
9. Examine missingness after cleaning.
10. Save the cleaned dataset for the analytical-table stage.

### Critical methodological rule

A textual unavailable marker such as `-` must not automatically be converted to zero.

For this project:

- `0` = an explicitly recorded zero;
- `-` = unavailable/unreported information;
- `NaN` = missing information.

Therefore, unavailable surveillance observations remain missing after cleaning.

This prevents the creation of artificial negative observations for subsequent machine-learning modelling.

In [1]:

# ============================================================
# NOTEBOOK 02 — WAHIS DATA CLEANING
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 1. DEFINE PROJECT PATHS
# ------------------------------------------------------------
# The notebook is normally executed from the notebooks/
# directory. We therefore move one level upward to identify
# the project root.
# ------------------------------------------------------------

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


RAW_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Quantitative data 2026-10-01.csv"
)

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEANED_DATA_PATH = (
    PROCESSED_DIR
    / "wahis_quantitative_cleaned.csv"
)


# ------------------------------------------------------------
# 2. LOAD THE RAW DATA
# ------------------------------------------------------------
# IMPORTANT:
# We load the original file again rather than reusing a
# potentially modified DataFrame from Notebook 01.
#
# This guarantees that the cleaning stage always starts
# from the original raw source.
# ------------------------------------------------------------

df_raw = pd.read_csv(RAW_DATA_PATH)

print("Raw dataset loaded successfully.")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")


# ------------------------------------------------------------
# 3. CREATE A WORKING COPY
# ------------------------------------------------------------
# The raw DataFrame is never modified.
#
# All cleaning operations are performed on df_clean.
# ------------------------------------------------------------

df_clean = df_raw.copy()


# ------------------------------------------------------------
# 4. STANDARDISE COLUMN NAMES
# ------------------------------------------------------------
# Convert column names into a consistent snake_case format.
#
# Examples:
#
#     "New outbreaks"
#         ↓
#     "new_outbreaks"
#
#     "Administrative Division"
#         ↓
#     "administrative_division"
#
# This makes the dataset easier to use in Python, SQL and
# machine-learning pipelines.
# ------------------------------------------------------------

def standardise_column_name(column):
    """Convert a column name into a simple snake_case format."""
    
    column = str(column).strip().lower()
    
    # Replace separators and punctuation with spaces.
    column = column.replace("/", "_")
    column = column.replace("-", "_")
    column = column.replace(" ", "_")
    
    # Remove repeated underscores.
    column = "_".join(
        part for part in column.split("_")
        if part
    )
    
    return column


df_clean.columns = [
    standardise_column_name(column)
    for column in df_clean.columns
]


print("\nStandardised column names:")
print(df_clean.columns.tolist())


# ------------------------------------------------------------
# 5. IDENTIFY TEXTUAL UNAVAILABLE MARKERS
# ------------------------------------------------------------
# The raw WAHIS extract contains "-" in some fields.
#
# We convert "-" to NaN because it represents unavailable
# information rather than an observed numerical zero.
#
# We deliberately DO NOT replace "-" with 0.
# ------------------------------------------------------------

UNAVAILABLE_MARKERS = [
    "-",
    "–",
    "—",
    "",
    "NA",
    "N/A",
    "na",
    "n/a",
    "NULL",
    "null",
]

df_clean = df_clean.replace(
    UNAVAILABLE_MARKERS,
    np.nan
)


# ------------------------------------------------------------
# 6. DEFINE QUANTITATIVE SURVEILLANCE VARIABLES
# ------------------------------------------------------------
# These variables describe reported disease-event impacts.
#
# They should be numeric after cleaning.
# ------------------------------------------------------------

numeric_columns = [
    "year",
    "new_outbreaks",
    "susceptible",
    "cases",
    "killed_and_disposed_of",
    "slaughtered",
    "deaths",
    "vaccinated",
]


# ------------------------------------------------------------
# 7. CONVERT QUANTITATIVE VARIABLES TO NUMERIC
# ------------------------------------------------------------
# errors="coerce" converts values that cannot be interpreted
# as numbers into NaN.
#
# This is preferable to forcing invalid values into zero.
# ------------------------------------------------------------

for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(
            df_clean[column],
            errors="coerce"
        )


# ------------------------------------------------------------
# 8. STANDARDISE SELECTED TEXT FIELDS
# ------------------------------------------------------------
# Remove unnecessary leading/trailing whitespace from
# categorical fields.
#
# Missing values remain missing.
# ------------------------------------------------------------

categorical_columns = [
    "semester",
    "world_region",
    "country",
    "administrative_division",
    "disease",
    "serotype_subtype_genotype",
    "animal_category",
    "species",
    "measuring_units",
]

for column in categorical_columns:
    if column in df_clean.columns:
        df_clean[column] = (
            df_clean[column]
            .astype("string")
            .str.strip()
        )


# ------------------------------------------------------------
# 9. STANDARDISE COUNTRY VALUES
# ------------------------------------------------------------
# The project is focused on Nigeria.
#
# We standardise the country field without changing the
# underlying meaning of the records.
# ------------------------------------------------------------

if "country" in df_clean.columns:
    df_clean["country"] = (
        df_clean["country"]
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )


# ------------------------------------------------------------
# 10. STANDARDISE SEMESTER VALUES
# ------------------------------------------------------------
# The raw dataset uses labels such as:
#
#     Jan-Jun 2005
#     Jul-Dec 2005
#
# We preserve the original reporting meaning but remove
# unnecessary whitespace.
# ------------------------------------------------------------

if "semester" in df_clean.columns:
    df_clean["semester"] = (
        df_clean["semester"]
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
        .str.strip()
    )


# ------------------------------------------------------------
# 11. CHECK YEAR VALIDITY
# ------------------------------------------------------------
# The year should be a valid four-digit reporting year.
#
# We do not silently delete invalid records here.
# Instead, we identify them for validation.
# ------------------------------------------------------------

invalid_years = df_clean[
    df_clean["year"].notna()
    & ~df_clean["year"].between(1900, 2100)
]

print(
    f"\nInvalid year records: "
    f"{len(invalid_years):,}"
)


# ------------------------------------------------------------
# 12. CHECK DUPLICATE RECORDS
# ------------------------------------------------------------
# Duplicate rows can artificially increase the apparent
# frequency of disease observations.
#
# We report duplicates before deciding how they should be
# handled in the analytical-table stage.
# ------------------------------------------------------------

duplicate_count = df_clean.duplicated().sum()

print(
    f"Duplicate complete rows: "
    f"{duplicate_count:,}"
)


# ------------------------------------------------------------
# 13. CHECK MISSINGNESS AFTER CLEANING
# ------------------------------------------------------------
# This provides an updated view of missing values after
# converting textual unavailable markers into NaN.
# ------------------------------------------------------------

missing_after_cleaning = (
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\nMissing values after cleaning:")
display(
    missing_after_cleaning[
        missing_after_cleaning > 0
    ]
)


# ------------------------------------------------------------
# 14. VERIFY THAT ZERO VALUES WERE PRESERVED
# ------------------------------------------------------------
# This is an important methodological check.
#
# Explicit zeros in the original data should remain zeros.
# They must not be confused with values that were converted
# from "-" to NaN.
# ------------------------------------------------------------

surveillance_columns_clean = [
    "new_outbreaks",
    "susceptible",
    "cases",
    "killed_and_disposed_of",
    "slaughtered",
    "deaths",
    "vaccinated",
]

print("\nExplicit zero counts in surveillance variables:")

for column in surveillance_columns_clean:
    if column in df_clean.columns:
        zero_count = (
            df_clean[column]
            .eq(0)
            .sum()
        )

        print(
            f"{column}: {zero_count:,}"
        )


# ------------------------------------------------------------
# 15. DISPLAY CLEANED DATA
# ------------------------------------------------------------
# Show a sample of the cleaned dataset for visual inspection.
# ------------------------------------------------------------

print("\nCleaned dataset preview:")
display(df_clean.head())


# ------------------------------------------------------------
# 16. FINAL DATASET SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANED WAHIS DATASET SUMMARY")
print("=" * 60)

print(f"Rows: {df_clean.shape[0]:,}")
print(f"Columns: {df_clean.shape[1]:,}")

print(
    f"Duplicate complete rows: "
    f"{df_clean.duplicated().sum():,}"
)

print(
    f"Missing administrative divisions: "
    f"{df_clean['administrative_division'].isna().sum():,}"
)

print(
    f"Missing species: "
    f"{df_clean['species'].isna().sum():,}"
)

print(
    f"Missing serotype/subtype/genotype: "
    f"{df_clean['serotype_subtype_genotype'].isna().sum():,}"
)

print("=" * 60)


# ------------------------------------------------------------
# 17. SAVE THE CLEANED DATASET
# ------------------------------------------------------------
# Save to data/processed rather than data/raw.
#
# This preserves the original WAHIS source file and creates
# a separate reproducible cleaned version.
# ------------------------------------------------------------

df_clean.to_csv(
    CLEANED_DATA_PATH,
    index=False
)

print(
    f"\nCleaned dataset saved to:\n"
    f"{CLEANED_DATA_PATH}"
)

Raw dataset loaded successfully.
Rows: 1,180
Columns: 19

Standardised column names:
['year', 'semester', 'world_region', 'country', 'administrative_division', 'disease', 'serotype_subtype_genotype', 'animal_category', 'event_id', 'species', 'outbreak_id', 'new_outbreaks', 'susceptible', 'measuring_units', 'cases', 'killed_and_disposed_of', 'slaughtered', 'deaths', 'vaccinated']

Invalid year records: 0
Duplicate complete rows: 0

Missing values after cleaning:


C:\Users\TECHGROW ANALYTICS\AppData\Local\Temp\ipykernel_21288\4026425208.py:151: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_clean = df_clean.replace(


outbreak_id                  1180
event_id                     1180
serotype_subtype_genotype     920
new_outbreaks                 735
vaccinated                    512
slaughtered                   491
susceptible                   468
killed_and_disposed_of        466
cases                         459
deaths                        456
measuring_units               446
species                       445
administrative_division         1
dtype: int64


Explicit zero counts in surveillance variables:
new_outbreaks: 0
susceptible: 0
cases: 1
killed_and_disposed_of: 686
slaughtered: 477
deaths: 52
vaccinated: 602

Cleaned dataset preview:


,year,semester,world_region,country,administrative_division,disease,serotype_subtype_genotype,animal_category,event_id,species,outbreak_id,new_outbreaks,susceptible,measuring_units,cases,killed_and_disposed_of,slaughtered,deaths,vaccinated
0,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Fowl pox (-2005),<NA>,Domestic,NaN,Birds,NaN,NaN,90.0,Animal,60.0,0.0,0.0,0.0,0.0
1,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Fowl typhoid,<NA>,Domestic,NaN,Birds,NaN,NaN,1000.0,Animal,800.0,0.0,0.0,25.0,0.0
2,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Infectious bursal disease (Gumboro disease),<NA>,Domestic,NaN,Birds,NaN,NaN,200.0,Animal,10.0,0.0,0.0,0.0,0.0
3,2005,Jul-Dec 2005,Africa,Nigeria,Nigeria,Newcastle disease virus (Inf. with),<NA>,Domestic,NaN,Birds,NaN,NaN,22100.0,Animal,3527.0,0.0,800.0,2076.0,0.0
4,2005,Jan-Jun 2005,Africa,Nigeria,Nigeria,Newcastle disease virus (Inf. with),<NA>,Domestic,NaN,Birds,NaN,NaN,8100.0,Animal,3430.0,0.0,0.0,912.0,13000.0



CLEANED WAHIS DATASET SUMMARY
Rows: 1,180
Columns: 19
Duplicate complete rows: 0
Missing administrative divisions: 1
Missing species: 445
Missing serotype/subtype/genotype: 920

Cleaned dataset saved to:
c:\Users\TECHGROW ANALYTICS\poultry-disease-early-warning-nigeria\data\processed\wahis_quantitative_cleaned.csv


In [2]:
# ============================================================
# NOTEBOOK 02 — CLEANING VALIDATION
# ============================================================

# Perform a final set of checks to confirm that the cleaning
# process produced the expected structure without changing
# the number of records.


# ------------------------------------------------------------
# Check 1 — Row count must remain unchanged
# ------------------------------------------------------------

assert len(df_clean) == len(df_raw), (
    "Row count changed during cleaning."
)


# ------------------------------------------------------------
# Check 2 — Column count must remain unchanged
# ------------------------------------------------------------

assert df_clean.shape[1] == df_raw.shape[1], (
    "Column count changed during cleaning."
)


# ------------------------------------------------------------
# Check 3 — The raw dataset must still contain the original
# textual '-' markers.
#
# This confirms that df_raw was not modified.
# ------------------------------------------------------------

raw_dash_count = (
    df_raw.astype("string")
    .eq("-")
    .sum()
    .sum()
)

print(
    f"'-' markers remaining in raw data: "
    f"{raw_dash_count:,}"
)


# ------------------------------------------------------------
# Check 4 — The cleaned dataset should no longer contain
# '-' as a missing-value marker.
# ------------------------------------------------------------

clean_dash_count = (
    df_clean.astype("string")
    .eq("-")
    .sum()
    .sum()
)

print(
    f"'-' markers remaining in cleaned data: "
    f"{clean_dash_count:,}"
)

assert clean_dash_count == 0, (
    "Some '-' markers remain in the cleaned dataset."
)


# ------------------------------------------------------------
# Check 5 — Confirm that the cleaned surveillance columns
# are numeric.
# ------------------------------------------------------------

print("\nCleaned surveillance data types:")

display(
    df_clean[
        surveillance_columns_clean
    ].dtypes
)


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print("\nCleaning validation completed successfully.")

'-' markers remaining in raw data: 6,393
'-' markers remaining in cleaned data: 0

Cleaned surveillance data types:


new_outbreaks             float64
susceptible               float64
cases                     float64
killed_and_disposed_of    float64
slaughtered               float64
deaths                    float64
vaccinated                float64
dtype: object


Cleaning validation completed successfully.


In [ ]:
# Cleaning Conclusion

The raw WAHIS dataset has been cleaned while preserving the distinction between explicitly reported zero values and unavailable or missing surveillance information.

Textual unavailable markers such as `-` were converted to missing values rather than zero. Quantitative surveillance variables were converted to numeric types, while categorical fields were standardised for consistency.

The original raw dataset remains unchanged, and the cleaned dataset has been saved separately under `data/processed/`.

The cleaned dataset is now ready for the next analytical stage:

**03 — WAHIS Analytical Table**

The analytical-table stage will determine which observations can legitimately be used for outbreak analysis and supervised machine-learning modelling, while continuing to avoid treating missing surveillance reports as confirmed disease absence.